In [1]:
from typing import Any, Dict, List, Optional
from pathlib import Path
import polars as pl
import art
from art.utils import trajectory_logging as art_logging
import os
import sys
from pathlib import Path


# import matplotlib.pyplot as plt
# import seaborn as sns

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)
    
# os.chdir(module_path)
print(f"Current Working Directory: {os.getcwd()}")

from src.utils.traj_utils import TrajectoryGroupLoader, combine_trajectory_groups

Current Working Directory: /home/guycoh/dac_test/AgentDaC/notebooks


## Bulk File Loading

In [ ]:
traj_directory = Path("/home/guycoh/dac_test/AgentDaC/.art/easy2hard_dac_guy/models/Qwen3-32B_08_27_21_48/trajectories/train")
file_paths = list(traj_directory.glob("*.jsonl"))
file_paths = [str(path) for path in file_paths]

df = combine_trajectory_groups(file_paths)


# df.write_parquet("all_train_trajectories.parquet")

In [6]:
df.write_parquet(traj_directory/"all_eval_trajectories.parquet")

In [14]:
from src.utils.replay import RewardBasedDoubleQuantileReplayBuffer

buffer = RewardBasedDoubleQuantileReplayBuffer(
    directory=str(traj_directory),
    grouping_keys=["index", "stop_depth"],
    quantile_fraction=0.2,
    buffer_size=70,  # Only keep the last N epoch files
)



Buffer size limit: using last 70 files out of 138 total files
Loading new trajectory file: 0092.jsonl
Loading new trajectory file: 0134.jsonl
Loading new trajectory file: 0113.jsonl
Loading new trajectory file: 0103.jsonl
Loading new trajectory file: 0107.jsonl
Loading new trajectory file: 0097.jsonl
Loading new trajectory file: 0093.jsonl
Loading new trajectory file: 0081.jsonl
Loading new trajectory file: 0088.jsonl
Loading new trajectory file: 0087.jsonl
Loading new trajectory file: 0073.jsonl
Loading new trajectory file: 0135.jsonl
Loading new trajectory file: 0105.jsonl
Loading new trajectory file: 0085.jsonl
Loading new trajectory file: 0136.jsonl
Loading new trajectory file: 0084.jsonl
Loading new trajectory file: 0080.jsonl
Loading new trajectory file: 0094.jsonl
Loading new trajectory file: 0108.jsonl
Loading new trajectory file: 0075.jsonl
Loading new trajectory file: 0074.jsonl
Loading new trajectory file: 0120.jsonl
Loading new trajectory file: 0133.jsonl
Loading new trajec

In [28]:
epochs = []
for group in buffer.trajectory_groups:
    epoch = buffer._get_group_epoch(group)
    epochs.append(epoch)

epochs = set(epochs)
print(f"Found {len(epochs)} epochs: {sorted(epochs)}")

Found 20 epochs: ['0118', '0119', '0120', '0121', '0122', '0123', '0124', '0125', '0126', '0127', '0128', '0129', '0130', '0131', '0132', '0133', '0134', '0135', '0136', '0137']


## Individual File Loading

In [ ]:
loader = TrajectoryGroupLoader(f"../{traj_path}")

In [ ]:
df = loader.create_dataframe().drop('messages_and_choices', 'metrics', 'metadata')
df

In [ ]:
loader.get_metric_statistics()

In [ ]:
loader.get_group_statistics()

In [ ]:
df.write_parquet("output_005.parquet")
